# **Architecture Transformer**

**Objectif du TP :** comprendre comment on est passé des réseaux récurrents (RNN, LSTM) aux Transformers (GPT, BERT), en implémentant chaque architecture et en regardant ses limites.

| Partie | Architecture | Données |
|--------|--------------|---------|
| 1 | RNN | Sinusoïde |
| 2 | LSTM | Sinusoïde |
| 3 | Transformer | Sinusoïde |
| 4 | GPT | Texte (anglais) |
| 5 | BERT | Texte (français) |

Les parties 1, 2 et 3 utilisent **la même tâche** (prédire la valeur suivante d'une sinusoïde à partir des 10 précédentes). Les différences de résultat viennent donc seulement de l'architecture.

## **Partie 1 : RNN**

Un **RNN** lit une séquence dans l'ordre et garde une mémoire, l'**état caché** $h_t$, mise à jour à chaque instant :

$$h_t = \tanh(W_{ih}x_t + W_{hh}h_{t-1} + b)$$

Les mêmes poids sont réutilisés à chaque instant. C'est la première architecture qu'on met en pratique.

### Étape 1 : préparer les données

**Objectif :** créer une sinusoïde de 200 points, puis la découper en exemples par **fenêtre glissante** : 10 valeurs en entrée, la 11ᵉ comme cible (190 exemples). Le jeu est séparé **chronologiquement** : 150 exemples pour l'entraînement, 40 pour le test. On teste ainsi le modèle sur un « futur » qu'il n'a jamais vu.

Les données ont la forme `(séquences, longueur, variables)` = `(190, 10, 1)`, car c'est ce qu'attend un RNN.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(42)

# --------------------------------------------------
# 1. Créer les données
# --------------------------------------------------
periode = torch.linspace(0, 20, 200)
serie = torch.sin(periode)
longueur_sequence = 10
# Chaque entrée contient 10 valeurs.
# La cible est la valeur immédiatement suivante.
X = torch.stack([
    serie[i:i + longueur_sequence]
    for i in range(len(serie) - longueur_sequence)
])
y = torch.stack([
    serie[i + longueur_sequence]
    for i in range(len(serie) - longueur_sequence)
])
# Un RNN attend : (nombre de séquences, longueur, nombre de variables)
X = X.unsqueeze(-1)  # (190, 10, 1)
y = y.unsqueeze(-1)  # (190, 1)

# Découpage chronologique : début pour apprendre, fin pour tester.
X_train, y_train = X[:150], y[:150]
X_test, y_test = X[150:], y[150:]

**Objectif :** tracer la série pour vérifier qu'on obtient bien une sinusoïde avant de modéliser.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(periode, serie)
plt.xlabel("x")
plt.ylabel("y")

### Étape 2 : définir le modèle

**Objectif :** construire le réseau, la fonction de perte et l'optimiseur.

- `nn.RNN(input_size=1, hidden_size=16)` : lit la séquence valeur par valeur avec une mémoire de 16 nombres.
- `nn.Linear(16, 1)` : transforme le **dernier** état caché en une prédiction.
- `MSELoss` : c'est un problème de **régression**, on mesure l'écart entre valeur prédite et valeur réelle.
- `Adam(lr=0.01)` : met à jour les poids.

Dimensions : `(B, 10, 1)` → RNN → `(B, 10, 16)` → dernier état → `(B, 16)` → Linear → `(B, 1)`.

In [ ]:
# --------------------------------------------------
# 2. Définir le modèle
# --------------------------------------------------

class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=1,     # Une valeur à chaque instant
            hidden_size=16,   # Une mémoire composée de 16 nombres
            batch_first=True
        )

        self.sortie = nn.Linear(16, 1)

    def forward(self, x):
        # Si aucun état initial n'est fourni, PyTorch utilise des zéros.
        etats, _ = self.rnn(x)

        # etats : (nombre de séquences, 10, 16)
        dernier_etat = etats[:, -1, :]  # (nombre de séquences, 16)

        # Transformer la mémoire finale en une valeur prédite.
        return self.sortie(dernier_etat)


modele = SimpleRNN()
critere = nn.MSELoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)

### Étape 3 : entraîner et tester

**Objectif :** pendant 300 époques, prédire, mesurer l'erreur, calculer les gradients (`backward`) et corriger les poids (`step`). Ensuite, évaluer sur les 40 exemples de test pour vérifier que le modèle **généralise**.

**Résultat :** l'erreur d'entraînement baisse bien au fil des époques, et l'erreur de test est d'environ **2,8·10⁻⁵**. La première prédiction (-0,3601) est très proche de la vraie valeur (-0,3639).

In [ ]:
# --------------------------------------------------
# 3. Entraîner le modèle
# --------------------------------------------------

modele.train()

for epoch in range(300):
    predictions = modele(X_train)
    perte = critere(predictions, y_train)

    optimiseur.zero_grad()  # Effacer les anciens gradients
    perte.backward()       # Calculer les nouveaux gradients
    optimiseur.step()      # Mettre à jour les paramètres

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d} | Erreur : {perte.item():.6f}")


# --------------------------------------------------
# 4. Tester le modèle
# --------------------------------------------------

modele.eval()

with torch.no_grad():
    predictions = modele(X_test)
    erreur_test = critere(predictions, y_test)

print(f"\nErreur de test : {erreur_test.item():.6f}")
print(f"Valeur réelle  : {y_test[0].item():.4f}")
print(f"Valeur prédite : {predictions[0].item():.4f}")

**Objectif :** vérifier la forme des prédictions : `(40, 1)`, une valeur par exemple de test.

In [ ]:
predictions.shape

### Étape 4 : visualiser les prédictions

**Objectif :** superposer la courbe réelle (bleu) et la courbe prédite (rouge). Quand les deux sont presque confondues, le modèle a bien appris la forme de la sinusoïde.

In [ ]:
import matplotlib.pyplot as plt

# Convertir les tenseurs en tableaux NumPy
valeurs_reelles = y_test.detach().cpu().numpy().reshape(-1)
valeurs_predites = predictions.detach().cpu().numpy().reshape(-1)

# Tracer les deux courbes
plt.figure(figsize=(12, 5))

plt.plot(
    valeurs_reelles,
    label="Valeurs réelles",
    color="blue",
    linewidth=2
)

plt.plot(
    valeurs_predites,
    label="Prédictions du RNN",
    color="red",
    linestyle="--",
    linewidth=2
)

plt.title("RNN : valeurs réelles et prédictions")
plt.xlabel("Indice de l'échantillon de test")
plt.ylabel("Valeur")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Limites du RNN

Le RNN marche bien ici, mais la tâche est facile (10 pas, aucun bruit). Ses limites apparaissent sur des séquences plus longues :

- **Gradients qui disparaissent ou explosent** : la rétropropagation à travers le temps multiplie le gradient à chaque pas. Il devient quasi nul (ou énorme), et le réseau n'apprend plus l'effet des entrées lointaines.
- **Mémoire courte** : `h_t` est entièrement réécrit à chaque pas, donc les informations anciennes s'écrasent.
- **Calcul séquentiel** : `h_t` dépend de `h_{t-1}`, donc pas de parallélisation.

→ Le **LSTM** a été créé pour mieux gérer la mémoire.

## **Partie 2 : LSTM**

Un **LSTM** transmet **deux** états : l'état caché $h_t$ et l'**état de cellule** $c_t$ (la mémoire interne). Trois **portes** (valeurs entre 0 et 1) contrôlent l'information :

| Porte | Rôle |
|-------|------|
| Oubli $f_t$ | Quoi garder de l'ancienne mémoire |
| Entrée $i_t$ | Quoi ajouter à la mémoire |
| Sortie $o_t$ | Quoi exposer dans $h_t$ |

**Objectif de la cellule suivante :** refaire les 4 étapes (données, modèle, entraînement, test) en remplaçant seulement `nn.RNN` par `nn.LSTM`.

**Comparaison avec le RNN :**

| | RNN | LSTM |
|---|-----|------|
| États | `h` | `h` et `c` |
| Mécanisme | `tanh` direct | 3 portes + contenu candidat |
| Paramètres de la couche ici | 304 | 1 216 (4 fois plus) |

**Résultat :** erreur de test ≈ **3·10⁻⁶**, environ 10 fois plus faible que le RNN.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# --------------------------------------------------
# 1. Créer les données
# --------------------------------------------------

serie = torch.sin(torch.linspace(0, 20, 200))
longueur_sequence = 10

X = torch.stack([
    serie[i:i + longueur_sequence]
    for i in range(len(serie) - longueur_sequence)
])

y = torch.stack([
    serie[i + longueur_sequence]
    for i in range(len(serie) - longueur_sequence)
])

# Forme attendue : (nombre de séquences, longueur, variables)
X = X.unsqueeze(-1)  # (190, 10, 1)
y = y.unsqueeze(-1)  # (190, 1)

# Découpage chronologique
X_train, y_train = X[:150], y[:150]
X_test, y_test = X[150:], y[150:]


# --------------------------------------------------
# 2. Définir le modèle LSTM
# --------------------------------------------------

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,     # Une variable à chaque instant
            hidden_size=16,   # 16 unités cachées
            num_layers=1,     # Une seule couche LSTM
            batch_first=True
        )

        self.sortie = nn.Linear(16, 1)

    def forward(self, x):
        # Sans états initiaux fournis, h₀ et c₀ sont mis à zéro.
        etats, (h_final, c_final) = self.lstm(x)

        # etats : (nombre de séquences, 10, 16)
        dernier_etat = etats[:, -1, :]  # (nombre de séquences, 16)

        # Une prédiction par séquence
        return self.sortie(dernier_etat)


modele = SimpleLSTM()
critere = nn.MSELoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)


# --------------------------------------------------
# 3. Entraîner le modèle
# --------------------------------------------------

modele.train()

for epoch in range(300):
    predictions = modele(X_train)
    perte = critere(predictions, y_train)

    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d} | Erreur : {perte.item():.6f}")


# --------------------------------------------------
# 4. Tester le modèle
# --------------------------------------------------

modele.eval()

with torch.no_grad():
    predictions = modele(X_test)
    erreur_test = critere(predictions, y_test)

print(f"\nErreur de test : {erreur_test.item():.6f}")
print(f"Valeur réelle  : {y_test[0].item():.4f}")
print(f"Valeur prédite : {predictions[0].item():.4f}")

**Objectif :** même graphique que pour le RNN, pour comparer visuellement.

In [ ]:
import matplotlib.pyplot as plt

# Convertir les tenseurs en tableaux NumPy
valeurs_reelles = y_test.detach().cpu().numpy().reshape(-1)
valeurs_predites = predictions.detach().cpu().numpy().reshape(-1)

# Tracer les deux courbes
plt.figure(figsize=(12, 5))

plt.plot(
    valeurs_reelles,
    label="Valeurs réelles",
    color="blue",
    linewidth=2
)

plt.plot(
    valeurs_predites,
    label="Prédictions du LSTM",
    color="red",
    linestyle="--",
    linewidth=2
)

plt.title("LSTM : valeurs réelles et prédictions")
plt.xlabel("Indice de l'échantillon de test")
plt.ylabel("Valeur")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Bilan RNN / LSTM, et limites du LSTM

| Modèle | Erreur de test | 1ʳᵉ prédiction (réel = -0,3639) |
|--------|----------------|---------------------------------|
| RNN | ≈ 2,8·10⁻⁵ | -0,3601 |
| LSTM | ≈ 3·10⁻⁶ | -0,3622 |

Le LSTM est meilleur, mais c'est une seule exécution sur un problème jouet : son vrai avantage se voit sur des séquences **longues**.

**Limites qui restent :** calcul toujours séquentiel, toute l'information passée doit tenir dans un vecteur de taille fixe, et modèle plus lourd.

→ Le **Transformer** supprime la récurrence : l'**attention** relie directement toutes les positions.

## **Partie 3 : Transformer**

Un **Transformer** n'a pas de mémoire récurrente : chaque position regarde directement les autres grâce à l'**attention**.

À partir des entrées, on calcule trois ensembles de vecteurs : **Q** (ce que chaque position cherche), **K** (ce qu'elle propose) et **V** (l'information à combiner).

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

L'attention **multi-têtes** fait plusieurs de ces calculs en parallèle. Comme l'attention ne connaît pas l'ordre, on **ajoute l'information de position**.

**Objectif de la cellule suivante :** construire un mini-Transformer (encodeur seul), l'entraîner et le tester sur la même tâche.

| Étape de `forward` | Dimensions |
|--------------------|------------|
| Projection `Linear(1, 16)` | `(B, 10, 1)` → `(B, 10, 16)` |
| + position `Embedding(10, 16)` | `(B, 10, 16)` |
| Encodeur (auto-attention 2 têtes + feed-forward) | `(B, 10, 16)` |
| Dernière position | `(B, 16)` |
| `Linear(16, 1)` | `(B, 1)` |

**Résultat :** erreur de test ≈ **6·10⁻⁶**.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# --------------------------------------------------
# 1. Créer les données
# --------------------------------------------------

serie = torch.sin(torch.linspace(0, 20, 200))
longueur_sequence = 10

X = torch.stack([
    serie[i:i + longueur_sequence]
    for i in range(len(serie) - longueur_sequence)
])

y = torch.stack([
    serie[i + longueur_sequence]
    for i in range(len(serie) - longueur_sequence)
])

# Forme : (nombre de séquences, longueur, variables)
X = X.unsqueeze(-1)  # (190, 10, 1)
y = y.unsqueeze(-1)  # (190, 1)

# Découpage chronologique
X_train, y_train = X[:150], y[:150]
X_test, y_test = X[150:], y[150:]


# --------------------------------------------------
# 2. Définir le modèle Transformer
# --------------------------------------------------

class MiniTransformer(nn.Module):
    def __init__(self, longueur_max=10):
        super().__init__()

        dimension = 16

        # Transformer chaque valeur en un vecteur de 16 nombres.
        self.projection = nn.Linear(1, dimension)

        # Un vecteur appris pour chacune des 10 positions.
        self.positions = nn.Embedding(longueur_max, dimension)

        # Un bloc d'encodeur Transformer.
        couche = nn.TransformerEncoderLayer(
            d_model=dimension,       # Taille des représentations : 16
            nhead=2,                 # Deux têtes d'attention
            dim_feedforward=32,      # Taille interne du réseau feed-forward
            dropout=0.0,             # Désactivé pour simplifier l'exemple
            batch_first=True
        )

        self.encodeur = nn.TransformerEncoder(
            couche,
            num_layers=1
        )

        # Transformer la représentation finale en une prédiction.
        self.sortie = nn.Linear(dimension, 1)

    def forward(self, x):
        # x : (B, 10, 1)
        longueur = x.size(1)

        # 1. Projection des valeurs
        x = self.projection(x)  # (B, 10, 16)

        # 2. Ajout de l'information de position
        indices = torch.arange(longueur, device=x.device)
        positions = self.positions(indices)  # (10, 16)

        x = x + positions.unsqueeze(0)  # (B, 10, 16)

        # 3. Mise en relation des valeurs par auto-attention
        representations = self.encodeur(x)  # (B, 10, 16)

        # 4. Récupérer la représentation de la dernière position
        derniere_position = representations[:, -1, :]  # (B, 16)

        # 5. Prédire la valeur suivante
        return self.sortie(derniere_position)  # (B, 1)


modele = MiniTransformer(longueur_max=longueur_sequence)
critere = nn.MSELoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)


# --------------------------------------------------
# 3. Entraîner le modèle
# --------------------------------------------------

modele.train()

for epoch in range(300):
    predictions = modele(X_train)
    perte = critere(predictions, y_train)

    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d} | Erreur : {perte.item():.6f}")


# --------------------------------------------------
# 4. Tester le modèle
# --------------------------------------------------

modele.eval()

with torch.no_grad():
    predictions = modele(X_test)
    erreur_test = critere(predictions, y_test)

print(f"\nErreur de test : {erreur_test.item():.6f}")
print(f"Valeur réelle  : {y_test[0].item():.4f}")
print(f"Valeur prédite : {predictions[0].item():.4f}")

**Objectif :** même graphique (réel et prédit) pour le Transformer.

In [ ]:
import matplotlib.pyplot as plt

# Convertir les tenseurs en tableaux NumPy
valeurs_reelles = y_test.detach().cpu().numpy().reshape(-1)
valeurs_predites = predictions.detach().cpu().numpy().reshape(-1)

# Tracer les deux courbes
plt.figure(figsize=(12, 5))

plt.plot(
    valeurs_reelles,
    label="Valeurs réelles",
    color="blue",
    linewidth=2
)

plt.plot(
    valeurs_predites,
    label="Prédictions du Transformer",
    color="red",
    linestyle="--",
    linewidth=2
)

plt.title("Transformer : valeurs réelles et prédictions")
plt.xlabel("Indice de l'échantillon de test")
plt.ylabel("Valeur")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Comparaison RNN / LSTM / Transformer

| Modèle | Erreur de test |
|--------|----------------|
| RNN | ≈ 2,8·10⁻⁵ |
| LSTM | ≈ 3·10⁻⁶ |
| Transformer | ≈ 6·10⁻⁶ |

Les trois réussissent. Ici le Transformer n'est pas avantagé : la tâche est minuscule. Il brille sur de longues séquences et de gros volumes de données.

| Critère | RNN | LSTM | Transformer |
|---------|-----|------|-------------|
| Traitement | Pas à pas | Pas à pas | **En parallèle** |
| Mémoire | `h_t` | `h_t` + `c_t` (portes) | Attention sur toute la séquence |
| Dépendances lointaines | Difficile | Meilleur | **Accès direct** |
| Position | Implicite | Implicite | **À ajouter** |
| Coût | Linéaire | Linéaire | **Quadratique** |

**Limites du Transformer :** coût quadratique avec la longueur, besoin de beaucoup de données, dépendance à l'encodage de position.

## **Partie 4 : GPT**

On passe au langage avec un modèle **déjà entraîné** (GPT-2) : on ne l'entraîne pas, on l'utilise.

**Objectif de la cellule suivante :** charger le tokenizer (texte → identifiants) et GPT-2, encoder la phrase `"Artificial intelligence can help people"`, puis **générer la suite** token par token.

**Principe :** GPT est un **décodeur** avec une **attention causale** (chaque position ne voit que les précédentes). Il prédit le token suivant, l'ajoute au texte, puis recommence : c'est la **génération autorégressive**.

Le premier lancement télécharge le modèle, ce qui explique le temps d'attente et les avertissements Hugging Face (token, symlinks), qui ne sont pas bloquants. Le texte généré est fluide mais pas forcément exact : le modèle choisit des mots probables, il ne vérifie pas les faits.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --------------------------------------------------
# 1. Charger le tokenizer et le modèle préentraîné
# --------------------------------------------------

nom_modele = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForCausalLM.from_pretrained(nom_modele)

# GPT-2 ne possède pas de token de padding dédié.
tokenizer.pad_token = tokenizer.eos_token
modele.config.pad_token_id = tokenizer.pad_token_id

modele.eval()

# --------------------------------------------------
# 2. Transformer le texte en tokens
# --------------------------------------------------

# GPT-2 est principalement entraîné sur de l'anglais.
texte = "Artificial intelligence can help people"

entrees = tokenizer(
    texte,
    return_tensors="pt"
)

print("Dimensions de l'entrée :", entrees["input_ids"].shape)
# (1, nombre de tokens du texte)

# --------------------------------------------------
# 3. Générer la suite, un token après l'autre
# --------------------------------------------------

with torch.no_grad():
    sortie = modele.generate(
        **entrees,
        max_new_tokens=50,       # Générer au maximum 50 tokens
        do_sample=True,          # Tirage aléatoire des tokens
        temperature=0.8,         # Contrôler la diversité
        top_p=0.9,               # Limiter les candidats probables
        pad_token_id=tokenizer.pad_token_id
    )

# --------------------------------------------------
# 4. Reconvertir les tokens en texte
# --------------------------------------------------

texte_genere = tokenizer.decode(
    sortie[0],
    skip_special_tokens=True
)

print("\nTexte généré :")
print(texte_genere)

## **Partie 5 : BERT**

**Objectif de la cellule suivante :** utiliser BERT multilingue pour retrouver le mot masqué dans `Paris est la [MASK] de la France.`

**Résultat :** « capitale » arrive en tête (≈ **36 %**), devant « ville » (≈ 21 %). BERT a bien utilisé le contexte. L'avertissement « Some weights ... were not used » est normal : on ne charge que la partie utile à cette tâche.

**Comparaison GPT / BERT :**

| Critère | GPT | BERT |
|---------|-----|------|
| Bloc | Décodeur | Encodeur |
| Attention | Causale (regarde le passé) | Bidirectionnelle (gauche et droite) |
| Entraînement | Prédire le token suivant | Retrouver des tokens masqués |
| Point fort | **Générer** du texte | **Comprendre** le texte |

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

# --------------------------------------------------
# 1. Charger le tokenizer et le modèle préentraîné
# --------------------------------------------------

nom_modele = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForMaskedLM.from_pretrained(nom_modele)

modele.eval()

# --------------------------------------------------
# 2. Préparer un texte avec un token masqué
# --------------------------------------------------

texte = f"Paris est la {tokenizer.mask_token} de la France."

entrees = tokenizer(
    texte,
    return_tensors="pt"
)

print("Texte :", texte)
print("Dimensions de l'entrée :", entrees["input_ids"].shape)
# (1, nombre de tokens du texte)

# Repérer la position du masque dans la séquence
position_masque = (
    entrees["input_ids"][0] == tokenizer.mask_token_id
).nonzero(as_tuple=True)[0].item()

# --------------------------------------------------
# 3. Prédire le token masqué
# --------------------------------------------------

with torch.no_grad():
    sortie = modele(**entrees)

# sortie.logits : (1, nombre de tokens, taille du vocabulaire)
scores = sortie.logits[0, position_masque, :]

# Transformer les scores en probabilités
probabilites = torch.softmax(scores, dim=-1)

# Sélectionner les 5 tokens les plus probables
top_probabilites, top_indices = torch.topk(probabilites, k=5)

# --------------------------------------------------
# 4. Afficher les propositions
# --------------------------------------------------

print("\nPropositions de BERT :")

for probabilite, indice in zip(top_probabilites, top_indices):
    token = tokenizer.decode([indice.item()]).strip()

    phrase = texte.replace(tokenizer.mask_token, token)

    print(f"{probabilite.item():.2%} | {phrase}")

---
## **Conclusion**

| Architecture | Idée | Limite |
|--------------|------|--------|
| RNN | État caché mis à jour pas à pas | Gradients instables, mémoire courte |
| LSTM | Portes + état de cellule | Séquentiel, plus lourd |
| Transformer | Attention entre toutes les positions | Coût quadratique, besoin de données |
| GPT | Décodeur, attention causale | Ne voit que le passé |
| BERT | Encodeur bidirectionnel | Pas conçu pour générer |

Chaque architecture corrige un défaut de la précédente : RNN → LSTM (mémoire), LSTM → Transformer (séquentialité et dépendances lointaines). Sur la sinusoïde les trois modèles réussissent : les différences se voient surtout sur de longues séquences et de grands volumes de données.